# Setup Environment

In [1]:
!pip install kagglehub -qq

In [2]:
from google.colab import userdata
import os

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['CIVITAI_TOKEN'] = userdata.get('CIVITAI_TOKEN')
os.environ['KAGGLE_USERNAME'] = 'zeerafle'
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_KEY')

## Setup GCP Bucket

In [3]:
# Authenticate.
from google.colab import auth
auth.authenticate_user()

# Install Cloud Storage FUSE.
!echo "deb https://packages.cloud.google.com/apt gcsfuse-`lsb_release -c -s` main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list
!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!apt -qq update && apt -qq install gcsfuse

deb https://packages.cloud.google.com/apt gcsfuse-jammy main
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1022  100  1022    0     0  12885      0 --:--:-- --:--:-- --:--:-- 12936
OK
37 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: https://packages.cloud.google.com/apt/dists/gcsfuse-jammy/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
The following NEW packages will be installed:
  gcsfuse
0 upgraded, 1 newly installed, 0 to remove and 37 not upgraded.
Need to get 14.9 MB of archives.
After this operation, 0 B of additional disk space will be used.
Selecting previously unsele

You can mount an entire bucket, or a path location within that bucket.
The local path to mount it must exist.

In [4]:
# Mount a Cloud Storage bucket or location, without the gs:// prefix.
mount_path = "sitting-posture"  # or a location like "my-bucket/path/to/mount"
local_path = f"/mnt/gs/{mount_path}"

!mkdir -p {local_path}
!gcsfuse --implicit-dirs {mount_path} {local_path}

{"timestamp":{"seconds":1755161599,"nanos":302255697},"severity":"INFO","message":"Start gcsfuse/3.2.0 (Go version go1.24.5) for app \"\" using mount point: /mnt/gs/sitting-posture\n"}
{"timestamp":{"seconds":1755161599,"nanos":302291658},"severity":"INFO","message":"GCSFuse config","config":{"AppName":"","CacheDir":"","Debug":{"ExitOnInvariantViolation":false,"Fuse":false,"Gcs":false,"LogMutex":false},"DisableAutoconfig":false,"EnableAtomicRenameObject":true,"EnableGoogleLibAuth":false,"EnableHns":true,"EnableNewReader":true,"FileCache":{"CacheFileForRangeRead":false,"DownloadChunkSizeMb":200,"EnableCrc":false,"EnableODirect":false,"EnableParallelDownloads":false,"ExperimentalExcludeRegex":"","ExperimentalParallelDownloadsDefaultOn":true,"MaxParallelDownloads":24,"MaxSizeMb":-1,"ParallelDownloadsPerFile":16,"WriteBufferSize":4194304},"FileSystem":{"DirMode":"755","DisableParallelDirops":false,"ExperimentalEnableDentryCache":false,"ExperimentalEnableReaddirplus":false,"FileMode":"644",

In [5]:
# Then you can access it like a local path.
!ls -lh {local_path}

total 0
drwxr-xr-x 1 root root 0 Aug 14 08:53 data
drwxr-xr-x 1 root root 0 Aug 14 08:53 generated_data


In [6]:
!mkdir {local_path}/generated_data
!mkdir {local_path}/generated_data/data

mkdir: cannot create directory ‘/mnt/gs/sitting-posture/generated_data’: File exists
mkdir: cannot create directory ‘/mnt/gs/sitting-posture/generated_data/data’: File exists


### Alternative (Google Drive)

In [7]:
# from google.colab import drive
# drive.mount('/content/drive')

# Download Data

In [8]:
import kagglehub

data_path = kagglehub.dataset_download('zeerafle/sitting-posture')

100%|██████████| 3.94G/3.94G [03:13<00:00, 21.9MB/s]

Extracting files...


In [9]:
data_path

'/root/.cache/kagglehub/datasets/zeerafle/sitting-posture/versions/4'

In [10]:
!ls {data_path}

ergonomis  non-ergonomis


# Setup ComfyUI

In [11]:
!apt -y update -qq
!apt -y install -qq aria2

!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -qq -r requirements.txt
!git reset --hard

!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared-linux-amd64 && chmod 777 /content/cloudflared-linux-amd64

37 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: https://packages.cloud.google.com/apt/dists/gcsfuse-jammy/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
The following additional packages will be installed:
  libaria2-0 libc-ares2
The following NEW packages will be installed:
  aria2 libaria2-0 libc-ares2
0 upgraded, 3 newly installed, 0 to remove and 37 not upgraded.
Need to get 1,513 kB of archives.
After this operation, 5,441 kB of additional disk space will be used.
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 126387 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:

In [12]:
from huggingface_hub import hf_hub_download
import os

def download_huggingface(repo_id_filename: list[tuple], dir):
    dir = os.path.join("/content/ComfyUI/models", dir)
    for repo_id, filename in repo_id_filename:
        hf_hub_download(repo_id=repo_id, filename=filename, local_dir=dir)

In [13]:
huggingface_models = {
    # "checkpoints": [
    # ],
    "controlnet": [
        ("xinsir/controlnet-union-sdxl-1.0", "diffusion_pytorch_model_promax.safetensors"),
    ],
    "loras": [
        ("ByteDance/SDXL-Lightning", "sdxl_lightning_8step_lora.safetensors"),
    ]
}

In [14]:
for key, value in huggingface_models.items():
    download_huggingface(value, key)

(…)ffusion_pytorch_model_promax.safetensors:   0%|          | 0.00/2.51G [00:00<?, ?B/s]

sdxl_lightning_8step_lora.safetensors:   0%|          | 0.00/394M [00:00<?, ?B/s]

In [15]:
!pip install -qq civitdl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.4 MB/s eta 0:00:00


In [16]:
def download_civitai(urls: list[str], dir):
    dir = os.path.join("/content/ComfyUI/models", dir)
    for url in urls:
        !civitdl -k {os.getenv('CIVITAI_TOKEN')} {url} {dir}

In [17]:
civitai_models = {
    "checkpoints": [
        # juggernaut_XL
        "https://civitai.com/api/download/models/1759168?type=Model&format=SafeTensor&size=full&fp=fp16"
        # real dream
        "https://civitai.com/api/download/models/2053273?type=Model&format=SafeTensor&size=pruned&fp=fp16"
        # gonzalomo
        # "https://civitai.com/api/download/models/2052057?type=Model&format=SafeTensor&size=full&fp=fp16"
    ]
}

In [18]:
for key, value in civitai_models.items():
    download_civitai(value, key)

/bin/bash: line 1: /content/ComfyUI/models/checkpoints: Is a directory
Usage: civitdl [-h] [-s SORTER] [-i INT] [--nsfw-mode MODE] [-k [API_KEY]]
               [--with-prompt | --no-with-prompt]
               [--without-model | --no-without-model] [--limit-rate BYTE]
               [--retry-count INT] [--pause-time FLOAT] [--cache-mode MODE]
               [--strict-mode MODE] [--model-overwrite | --no-model-overwrite]
               [--with-color | --no-with-color] [--verbose | --no-verbose]
               [-v]
               srcmodels [srcmodels ...] rootdir
civitdl: Error: the following arguments are required: rootdir


In [19]:
# !ln -s {data_path} input/

In [20]:
%%bash

# Set up environment
COMFYUI_DIR=/content/ComfyUI

NODES=(
    "https://github.com/ltdrdata/ComfyUI-Manager"
    "https://github.com/cubiq/ComfyUI_essentials"
    "https://github.com/rgthree/rgthree-comfy"
    "https://github.com/Fannovel16/comfyui_controlnet_aux"
    "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes"
    "https://github.com/adieyal/comfyui-dynamicprompts"
    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts"
    "https://github.com/welltop-cn/ComfyUI-TeaCache"
)

function provisioning_get_nodes() {
    for repo in "${NODES[@]}"; do
        dir="${repo##*/}"
        path="${COMFYUI_DIR}/custom_nodes/${dir}"
        requirements="${path}/requirements.txt"
        if [[ -d $path ]]; then
            if [[ ${AUTO_UPDATE,,} != "false" ]]; then
                printf "Updating node: %s...\n" "${repo}"
                ( cd "$path" && git pull )
                if [[ -e $requirements ]]; then
                   pip install -qq --no-cache-dir -r "$requirements"
                fi
            fi
        else
            printf "Downloading node: %s...\n" "${repo}"
            git clone "${repo}" "${path}" --recursive
            if [[ -e $requirements ]]; then
                pip install -qq --no-cache-dir -r "${requirements}"
            fi
        fi
    done
}

# Download and install nodes
echo "Starting node downloads..."
echo "=========================="
provisioning_get_nodes


Starting node downloads...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 416.5/416.5 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.0/182.0 kB 319.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 185.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 356.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 856.7/856.7 kB 358.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 282.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 131.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 245.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.7/54.7 kB 285.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.9/913.9 kB 122.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 247.0 MB/s eta 0:00:00
     ━━━━━━

Cloning into '/content/ComfyUI/custom_nodes/ComfyUI-Manager'...
Cloning into '/content/ComfyUI/custom_nodes/ComfyUI_essentials'...
Cloning into '/content/ComfyUI/custom_nodes/rgthree-comfy'...
Cloning into '/content/ComfyUI/custom_nodes/comfyui_controlnet_aux'...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.8 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.8 which is incompatible.
Cloning into '/content/ComfyUI/custom_nodes/ComfyUI_Comfyroll_CustomNodes'...
Cloning into '/content/ComfyUI/custom_nodes/comfyui-dynamicprompts'...
Cloning into '/content/ComfyUI/custom_nodes/ComfyUI-Custom-Scripts'...
Cloning into '/content/ComfyUI/custom_nodes/ComfyUI-TeaCache'...


In [21]:
import atexit, requests, subprocess, time, re, os
from random import randint
from threading import Timer
from queue import Queue
def cloudflared(port, metrics_port, output_queue):
    atexit.register(lambda p: p.terminate(), subprocess.Popen(['/content/cloudflared-linux-amd64', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--metrics', f'127.0.0.1:{metrics_port}'], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT))
    attempts, tunnel_url = 0, None
    while attempts < 10 and not tunnel_url:
        attempts += 1
        time.sleep(3)
        try:
            tunnel_url = re.search("(?P<url>https?:\/\/[^\s]+.trycloudflare.com)", requests.get(f'http://127.0.0.1:{metrics_port}/metrics').text).group("url")
        except:
            pass
    if not tunnel_url:
        raise Exception("Can't connect to Cloudflare Edge")
    output_queue.put(tunnel_url)
output_queue, metrics_port = Queue(), randint(8100, 9000)
thread = Timer(2, cloudflared, args=(8188, metrics_port, output_queue))
thread.start()
thread.join()
tunnel_url = output_queue.get()
os.environ['webui_url'] = tunnel_url
print(tunnel_url)

https://creative-dayton-arnold-quantity.trycloudflare.com


In [ ]:
%cd /content/ComfyUI
# !python main.py --dont-print-server --output-directory "/content/drive/MyDrive/ComfyUI/output"
!python main.py --dont-print-server --output-directory "/mnt/gs/sitting-posture/generated_data/data"

/content/ComfyUI
Setting output directory to: /mnt/gs/sitting-posture/generated_data/data
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2025-08-14 09:00:17.773
** Platform: Linux
** Python version: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
** Python executable: /usr/bin/python3
** ComfyUI Path: /content/ComfyUI
** ComfyUI Base Folder Path: /content/ComfyUI
** User directory: /content/ComfyUI/user
** ComfyUI-Manager config path: /content/ComfyUI/user/default/ComfyUI-Manager/config.ini
** Log path: /content/ComfyUI/user/comfyui.log

Prestartup times for custom nodes:
   0.0 seconds: /content/ComfyUI/custom_nodes/rgthree-comfy
   5.2 seconds: /content/ComfyUI/custom_nodes/ComfyUI-Manager

Checkpoint files will always be loaded safely.
Total VRAM 22693 MB, total RAM 54232 MB
pytorch version: 2.6.0+cu124
Set vram state to: NORMAL_VRAM
Device: cuda:0 NVIDIA L4 : cudaMallocAsync
Using pytorch attention
Python ver

# ComfyUI Script

In [ ]:
import json
from urllib import request

#This is the ComfyUI api prompt format.

#If you want it for a specific workflow you can "enable dev mode options"
#in the settings of the UI (gear beside the "Queue Size: ") this will enable
#a button on the UI to save workflows in api format.

#keep in mind ComfyUI is pre alpha software so this format will change a bit.

#this is the one for the default workflow
prompt_text = """
{
    "3": {
        "class_type": "KSampler",
        "inputs": {
            "cfg": 8,
            "denoise": 1,
            "latent_image": [
                "5",
                0
            ],
            "model": [
                "4",
                0
            ],
            "negative": [
                "7",
                0
            ],
            "positive": [
                "6",
                0
            ],
            "sampler_name": "euler",
            "scheduler": "normal",
            "seed": 8566257,
            "steps": 20
        }
    },
    "4": {
        "class_type": "CheckpointLoaderSimple",
        "inputs": {
            "ckpt_name": "v1-5-pruned-emaonly.safetensors"
        }
    },
    "5": {
        "class_type": "EmptyLatentImage",
        "inputs": {
            "batch_size": 1,
            "height": 512,
            "width": 512
        }
    },
    "6": {
        "class_type": "CLIPTextEncode",
        "inputs": {
            "clip": [
                "4",
                1
            ],
            "text": "masterpiece best quality girl"
        }
    },
    "7": {
        "class_type": "CLIPTextEncode",
        "inputs": {
            "clip": [
                "4",
                1
            ],
            "text": "bad hands"
        }
    },
    "8": {
        "class_type": "VAEDecode",
        "inputs": {
            "samples": [
                "3",
                0
            ],
            "vae": [
                "4",
                2
            ]
        }
    },
    "9": {
        "class_type": "SaveImage",
        "inputs": {
            "filename_prefix": "ComfyUI",
            "images": [
                "8",
                0
            ]
        }
    }
}
"""

def queue_prompt(prompt):
    p = {"prompt": prompt}

    # If the workflow contains API nodes, you can add a Comfy API key to the `extra_data`` field of the payload.
    # p["extra_data"] = {
    #     "api_key_comfy_org": "comfyui-87d01e28d*******************************************************"  # replace with real key
    # }
    # See: https://docs.comfy.org/tutorials/api-nodes/overview
    # Generate a key here: https://platform.comfy.org/login

    data = json.dumps(p).encode('utf-8')
    req =  request.Request("http://127.0.0.1:8188/prompt", data=data)
    request.urlopen(req)


with open(f"{local_path}/generated_data/sdxl_controlnet.json", 'r') as f:
    workflow = json.load(f)
# #set the text prompt for our positive CLIPTextEncode
# prompt["6"]["inputs"]["text"] = "masterpiece best quality man"

# #set the seed for our KSampler node
# prompt["3"]["inputs"]["seed"] = 5


# queue_prompt(prompt)



NameError: name 'local_path' is not defined

In [ ]:
workflow

In [ ]:
# Assuming 'workflow' is the dictionary containing the workflow data

# Access the node with id 81
node_81 = None
for node in workflow['nodes']:
    if node['id'] == 81:
        node_81 = node
        break

if node_81:
    # Access the 'widgets_values' list
    widgets_values = node_81['widgets_values']

    # The string you want to modify is the first element in the list
    prompt_string = widgets_values[0]

    print(f"Original string: {prompt_string}")

    # Now you can modify the string dynamically.
    # For example, let's change "front view" to "side view"
    modified_string = prompt_string.replace("front view", "side view")

    # Update the widgets_values list with the modified string
    widgets_values[0] = modified_string

    print(f"Modified string: {widgets_values[0]}")

    # The 'workflow' dictionary is now updated with the change
    # You can now use the updated 'workflow' for further processing
else:
    print("Node with ID 81 not found.")